In [ ]:
# Minimal CNN + LSTM text classifier (toy example)
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense
from tensorflow.keras.optimizers import Adam

# -- Toy dataset --
sentences = [
    "I love this product, it works great",
    "Absolutely fantastic service",
    "This is the best I've used",
    "Very satisfied and happy",
    "I hate this, it broke immediately",
    "Terrible experience, would not recommend",
    "Awful, I am disappointed",
    "Not good, very poor quality"
]

labels = np.array([1, 1, 1, 1, 0, 0, 0, 0])  # 1=good, 0=bad

# -- Preprocessing --
vocab_size = 1000    # only the 1000 most frequent unique words existing in the train set are included in the vocab, and assigned an index
max_len = 20         # every input sentence, regardless of its size, will be represented by a vector of 20 integer indices, 
                     # sentence shorter than 20 will be padded with 0
tokenizer = Tokenizer(num_words=vocab_size, 
                      oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)

X = pad_sequences(sequences, 
                  maxlen=max_len, 
                  padding='post')

# -- Build model (Embedding -> Conv1D -> MaxPool -> LSTM -> Dense) --
embedding_dim = 50
model = Sequential([
    Embedding(input_dim=vocab_size, 
              output_dim=embedding_dim, 
              input_length=max_len),
    Conv1D(filters=128, 
           kernel_size=5, 
           activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(64),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer=Adam(learning_rate=1e-3), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# Assuming max_len = 20, the input shape is (None, 20)
# 'None' represents the batch size, which can be variable.
input_shape_tuple = (None, max_len)

# Force the model to build its weights based on the expected input shape
model.build(input_shape=input_shape_tuple)

model.summary()

# -- Train (very small toy training for demo) --
history = model.fit(X, labels, epochs=12, batch_size=2, verbose=2)

# -- Helper to classify a single sentence --
def classify_sentence(sentence, threshold=0.5):
    seq = tokenizer.texts_to_sequences([sentence])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    prob = float(model.predict(pad, verbose=0)[0,0])
    label = 1 if prob >= threshold else 0
    return {'sentence': sentence, 'probability': prob, 'label': label}

# -- Demo predictions --
tests = [
    "I am very happy with this",
    "What a horrible product",
    "Mediocre, not good movie"
]
for t in tests:
    print(classify_sentence(t))